In [ ]:
from scapy.all import *
import ipaddress # usado para redes tipo 172.19.0.0/29

def craft_discovery_pkts(protocolos, ips, num_pkts=None, port=80):
    """
    Funcion que crea paquetes de descubrimiento de hosts activos usando ARP (si es local, si aplica) e ICMP, TCP y UDP.

    Parametros:

    - protocolos: protocolos que se vaya a utilizar para el escaneo. Ej: ["ICMP"].
    - ips: IP o red de destino (ej: "10.0.1.10" o "10.0.1.0/24").
    - num_pkts: numero de paquetes por protocolo (si no se indica, se envia solo 1 por protocolo)
    - port: puerto destino para TCP y UDP (por defecto lo dejo en 80).

    Funcionamiento: 

    ARP: funciona a nivel de redes locales, para resoluciones de direcciones MAC (nivel 2).
    ICMP: envia Timestamp Request para detectar los hosts activos.
    TCP: envia paquetes con flag ACK para provocar respuesta RST.
    UDP: envia datagramas y detecta respuesta ICMP si el puerto esta cerrado.

    Devuelve:

    Devuelve una lista de paquetes Scapy listos para ser enviados.

    """

    packets=[] # listado de paquetes que se van a enviar

    # si protocolos es string lo convierto a lista
    if isinstance(protocolos,str):

        protocolos=[protocolos]

    # si no se especifica el numero de paquetes
    if num_pkts is None:

        num_pkts={}

        for proto in protocolos:

            num_pkts[proto]=1 # envio uno por protocolo


    # ICMP TIMESTAMP
    if "ICMP" in protocolos: # si esta activo ICMP

        for i in range(num_pkts["ICMP"]): # repite el envio N veces dependiendo de "num_pkts"

            l3 = IP(dst=ips) # capa 3, IP de destino (siendo posible IP o red)

            l4 = ICMP(type=13) # capa 4, ICMP Timestamp Request, para ver si el host responde ICMP

            pkt = l3 / l4 # construyo paquete

            packets.append(pkt) # guardo paquete


    # UDP DISCOVERY
    if "UDP" in protocolos:

        for i in range(num_pkts["UDP"]):

            l3 = IP(dst=ips) 

            l4 = UDP(dport=port) # envio UDP al puerto 80 o el que se pase.

            pkt = l3 / l4

            packets.append(pkt)


    # TCP ACK DISCOVERY
    if "TCP" in protocolos:

        for i in range(num_pkts["TCP"]):

            l3 = IP(dst=ips)

            l4 = TCP(dport=port,flags="A") # es ACK debido al flag que la flag es la A

            pkt = l3 / l4

            packets.append(pkt)


    return packets # devuelvo todos los paquetes listos



protocolos=["ICMP","TCP","UDP"]

target_ips=["172.19.0.2","172.19.0.3","10.0.1.10","10.0.2.10"] # lista de objetivos a examinar

red_local = "172.19.0.0/29" # defino cuales son locales, decido si se usa ARP

ans = [] # lista para guardar las respuestas

for ip in target_ips: # bucle para analizar una ip de una 

    # si es local uso ARP
    if ipaddress.ip_address(ip) in ipaddress.ip_network(red_local): # miro si la ip esta dentro de la red local

        arp = Ether(dst="ff:ff:ff:ff:ff:ff") / ARP(pdst=ip) # broadcast ARP

        r = srp(arp, timeout=2, verbose=0, iface="br-3f020f3d3088")[0] # capa 2, envio a esa capa y se descubre los host locales reales
        ans += r # guardo las respuestas

    # si no es local 
    else:

        pkts = craft_discovery_pkts(protocolos, ip) # genero paquetes ICMP, TCP y UDP

        for pkt in pkts:
            r = sr(pkt, timeout=1, verbose=0)[0] # envio a capa 3
            ans += r

hosts_activos = set() # evito los duplicados con set

for snd, rcv in ans: # snd es lo que envie, y rcv es la respuesta

    if rcv.haslayer(ARP): # si es ARP
        hosts_activos.add(rcv.psrc) # responde con la Ip
    else:
        hosts_activos.add(snd[IP].dst) # si no lo es, responde con la IP de destino original

print("Hosts activos detectados:")

for ip in hosts_activos: # muestro los hosts activos sin duplicado
    print(ip)